# 01 · Bronze ingestion

Read telemetry JSONL, the fleet SQLite database, and real weather snapshots/API responses. Save their data before Silver cleaning. Fleet and telemetry are synthetic.

**Run order:** Bronze → Silver → Gold → separate Power BI report. Bronze calls the demo preparation notebook in Databricks; you do not need to run setup twice. Reruns overwrite only this demo namespace. All dates use UTC.

## Paths, dates and imports

In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
import json
import os
import sqlite3
import uuid

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

project_dir = Path.cwd()
if not (project_dir / "weather.py").exists():
    project_dir = project_dir.parent
project_dir = Path(os.environ.get("FLEET_PROJECT_DIR", project_dir)).resolve()
source_dir = project_dir / "data" / "sources"
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)
run_id = str(uuid.uuid4())

# Databricks already supplies spark. Local Python needs a Delta-enabled session.
if "spark" not in globals():
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.master("local[2]").appName("fleet-activity")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "2"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    spark.sparkContext.setLogLevel("ERROR")
print("Report period:", start_date, "to", end_date)

:: loading settings :: url = jar:file:<repo>/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: <home>/.ivy2.5.2/cache
The jars for the packages stored in: <home>/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09d6dc2a-793c-4b50-a882-f65051dc57ae;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central


	found org.apache.logging.log4j#log4j-core;2.25.3 in central
:: resolution report :: resolve 180ms :: artifacts dl 27ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	io.delta#delta-spark_4.1_2.13;4.1.0 from central in [default]
	io.delta#delta-storage;4.1.0 from central in [default]
	io.unitycatalog#unitycatalog-client;0.4.0 from central in [default]
	org.apache.logging.log4j#log4j-api;2.25.3 from central in [default]
	org.apache.logging.log4j#log4j-core;2.25.3 from central in [default]
	org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 from central in [default]
	org.slf4j#slf4j-api;2.0.13 from central in [default]
	:: evicted modules:
	org.slf4j#slf4j-api;2.0.17 by [org.slf4j#slf4j-api;2.0.13] in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Report period: 2026-08-01 to 2026-08-30


## Read telemetry into Bronze
Read the timestamp as text so a bad timestamp can be inspected later. PERMISSIVE keeps broken JSON in `_corrupt_record`. Save before querying that column. Expected: 4,316 rows.

In [2]:
telemetry_schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("odometer_km", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True),
])
raw_telemetry = (spark.read.schema(telemetry_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(str(source_dir / "telemetry" / "*.jsonl"))
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id)))
bronze_path = str(output_dir / "delta" / "bronze_telemetry")
raw_telemetry.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_path)
bronze = spark.read.format("delta").load(bronze_path)
bronze.printSchema()
print("Bronze rows:", bronze.count())

root
 |-- vehicle_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- run_id: string (nullable = true)



Bronze rows: 4316


## Read the fleet database
Extract the vehicle roster and depot coordinates. Silver will check the roster before Gold uses it.

In [3]:
database_path = source_dir / "fleet.sqlite"
with sqlite3.connect(database_path.as_uri() + "?mode=ro", uri=True) as database:
    fleet_rows = database.execute("""
        SELECT UPPER(TRIM(v.vehicle_id)), v.model, UPPER(TRIM(v.depot_id)),
               d.depot_name, d.latitude, d.longitude
        FROM vehicles AS v
        LEFT JOIN depots AS d ON v.depot_id = d.depot_id
        ORDER BY v.vehicle_id
    """).fetchall()
    depot_rows = database.execute("SELECT depot_id, latitude, longitude FROM depots ORDER BY depot_id").fetchall()
depots = []
for row in depot_rows:
    depot = {"depot_id": row[0], "latitude": row[1], "longitude": row[2]}
    depots.append(depot)
fleet = spark.createDataFrame(fleet_rows,
    "vehicle_id string, model string, depot_id string, depot_name string, latitude double, longitude double")

fleet.show(4, truncate=False)

+----------+-------+--------+----------+--------+---------+
|vehicle_id|model  |depot_id|depot_name|latitude|longitude|
+----------+-------+--------+----------+--------+---------+
|EV001     |Model A|CHI     |Chicago   |41.88   |-87.63   |
|EV002     |Model B|CHI     |Chicago   |41.88   |-87.63   |
|EV003     |Model A|CHI     |Chicago   |41.88   |-87.63   |
|EV004     |Model B|CHI     |Chicago   |41.88   |-87.63   |
+----------+-------+--------+----------+--------+---------+
only showing top 4 rows


## Save fleet Bronze

In [4]:
fleet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "bronze_fleet"))
print("Fleet rows saved:", fleet.count())

Fleet rows saved: 24


## Ingest weather
The helper saves raw API JSON and validates its response shape. This extracted Bronze table holds the daily fields. Silver checks depot/date coverage and uniqueness. Databricks uses saved real responses; the local notebook also supports live requests.

In [5]:
import sys
sys.path.insert(0, str(project_dir))
from weather import load_weather

offline = os.environ.get("FLEET_OFFLINE") == "1"
weather_dir = project_dir / ("examples/weather" if offline else "data/weather")
weather_rows, weather_snapshots = load_weather(
    depots, start_date, end_date, weather_dir,
    refresh=os.environ.get("FLEET_REFRESH_WEATHER") == "1", offline=offline)
weather = spark.createDataFrame(weather_rows,
    "depot_id string, event_date date, temperature_c double, precipitation_mm double")
weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(output_dir / "delta" / "bronze_weather"))
print("Weather rows saved:", weather.count())

Weather rows saved: 60


## Next: 02_silver_cleaning
Bronze tables are saved. Open Silver next. If Bronze fails, stop here and fix it before running downstream notebooks. Old outputs can remain after a failed run.